In [1]:
from pathlib import Path

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Vector database path
VECTOR_DB_DIR = Path.cwd() / "vector_store" / "faiss_index"

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = FAISS.load_local(
    str(VECTOR_DB_DIR),
    embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS Vector Database Loaded Successfully!")

C:\Users\harsh\AppData\Local\Temp\ipykernel_26020\1843868188.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
c:\Users\harsh\OneDrive\Desktop\Projects\RAGP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 839.83it/s]


FAISS Vector Database Loaded Successfully!


In [2]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("Retriever Created Successfully!")

Retriever Created Successfully!


In [3]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

print("Ollama LLM Initialized!")

Ollama LLM Initialized!


In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
You are a PDF-based AI assistant.

Answer the question using only the given context.
If the answer is not present in the context,
say that the information is not available in the PDF.

Context:
{context}

Question:
{question}

Answer:
""")

rag_chain = prompt | llm | StrOutputParser()

print("RAG Chain Created Successfully!")

RAG Chain Created Successfully!


In [8]:
%pip install gradio

Note: you may need to restart the kernel to use updated packages.


c:\Users\harsh\OneDrive\Desktop\Projects\RAGP\.venv\Scripts\python.exe: No module named pip


In [ ]:
import gradio as gr

def chat_with_pdf(question, history):

    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    response = rag_chain.invoke({
        "context": context,
        "question": question
    })

    return response

demo = gr.ChatInterface(
    fn=chat_with_pdf,
    title="PDF RAG Chatbot",
    description="Ask questions from your PDF using FAISS and Ollama."
)

demo.launch()